# 🧠 Multi-Drone Collective Intelligence Swarm (Colab Edition)

**SOTA Multi-Agent Reinforcement Learning with Emergent & Targeted Communication**

---

### 🌟 System Architecture Overview
1. **Pretrained Spatial Perception (Frozen Backbone)**: Features extracted via ConvNeXt + Transformer MHSA bottlenecks (`GlobalMapCNN` and `LocalCNN`) from the *Multi-Agent Systems* project, eliminating the need to retrain low-level vision from scratch.
2. **Specialized Agent Roles**: 
   - 🔭 **Scout**: Fast exploration, wide coverage, discovery bonuses.
   - 📦 **Carrier**: Payload transport & rendezvous at the target site.
   - 🔬 **Analyzer**: Target inspection & diagnostic hovering.
3. **TarMAC (Targeted Multi-Agent Communication)**: Multi-head attention-gated message passing ($m_i, s_i, q_i$). Drones learn **who** to talk to and **what** to say, subject to an L2 bandwidth penalty $-\lambda \|m_i\|_2$ to enforce communication economy ("don't talk unless necessary").
4. **QMIX Centralized Value Factorization**: Joint cooperative multi-agent value network conditioned on the global swarm state with non-negative monotonic hypernetworks.
5. **5-Stage Curriculum Training**: Automated progression from basic search to full multi-phase missions with spontaneous drone crashes and communication dropouts.

## 🛠️ Cell 1: Environment & Dependency Setup
Run this cell in Google Colab to clone the repository into the Colab VM, install required packages, and configure paths.

In [ ]:
# 1. Setup repository for Google Colab
import os
import sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("[Colab] Setting up repository in VM...")
    if not os.path.exists("swarm_collective"):
        if not os.path.exists("oziz-is-drosophil"):
            !git clone https://github.com/Kairatzh/oziz-is-drosophil.git
        %cd oziz-is-drosophil
    if os.getcwd() not in sys.path:
        sys.path.insert(0, os.getcwd())
else:
    if os.getcwd() not in sys.path:
        sys.path.insert(0, os.getcwd())

# 2. Install dependencies
!pip install -q torch torchvision gymnasium matplotlib pyyaml huggingface_hub tqdm pygame

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Device Check] Using compute device: {device}")
if torch.cuda.is_available():
    print(f"[GPU Info] {torch.cuda.get_device_name(0)}")
print(f"[Working Directory] {os.getcwd()}")
print("[Setup] Ready to run!")

## 📦 Cell 2: Download Pretrained Perception Weights
Downloads the trained ConvNeXt + Transformer checkpoint (`mas_50k_dr_faults_ep50000.pt`) from Hugging Face into `pretrained/`.

In [ ]:
import os
from huggingface_hub import hf_hub_download

os.makedirs("pretrained", exist_ok=True)
ckpt_path = "pretrained/mas_backbone.pt"

if not os.path.exists(ckpt_path):
    print("[Hugging Face] Downloading pretrained backbone weights (approx. 270 MB)...")
    try:
        downloaded = hf_hub_download(
            repo_id="simoswish/MAS_MultiDroneExploration",
            filename="mas_50k_dr_faults_ep50000.pt",
            local_dir="pretrained/"
        )
        os.rename(downloaded, ckpt_path)
        print(f"[Hugging Face] ✅ Checkpoint successfully saved to: {ckpt_path}")
    except Exception as e:
        print(f"[Notice] Download skipped or failed ({e}). Backbone will initialize freshly for training.")
else:
    print(f"[Hugging Face] ✅ Found existing pretrained weights at: {ckpt_path}")

## 🧪 Cell 3: Environment & Agent Architecture Smoke-Test
Validates that:
- `SwarmEnv` correctly initializes all 3 roles (Scouts, Carriers, Analyzers).
- Backbone feature extractor maps observation channels to 1024-dim and 384-dim latent representations.
- TarMAC communication produces valid attention keys and messages.

In [ ]:
import os
import sys
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
if not os.path.exists("swarm_collective") and os.path.exists("oziz-is-drosophil"):
    %cd oziz-is-drosophil
    sys.path.insert(0, os.getcwd())

from swarm_collective.env import SwarmEnv, ROLE_NAMES
from swarm_collective.agents import build_backbone, SwarmAgent, SwarmSystem

# 1. Initialize environment with 4 heterogeneous drones
env = SwarmEnv(
    grid_size=32,
    n_scouts=2,
    n_carriers=1,
    n_analyzers=1,
    task="full_pipeline",
    comm_range=5,
    vision_radius=3
)

obs_dict, info = env.reset()
print(f"=== Swarm Environment Initialized ===")
print(f"Total Drones : {len(env.role_ids)}")
print(f"Roles        : {[ROLE_NAMES[r] for r in env.role_ids]}")
print(f"Target Pos   : {env.target_pos}")
print(f"Mission Phase: {info.get('task_phase', 'search')}")

# 2. Build Backbone & Swarm Agent
backbone = build_backbone(
    ckpt_path=ckpt_path if os.path.exists(ckpt_path) else None,
    freeze=True
)

agent = SwarmAgent(
    backbone=backbone,
    role_embedding_dim=64,
    hidden_dim=512,
    dim_msg=16,
    dim_key=8,
    comm_cost=0.01
)
system = SwarmSystem(agent=agent).to(device)

print("\n✅ Swarm Agent and Backbone loaded successfully!")

## 📡 Cell 4: TarMAC Targeted Attention & Communication Analysis
Tests the attention mechanism. Agents within communication range dynamically attend to each other's broadcasted signatures. If a drone is out of range, its link is masked.

In [ ]:
import numpy as np
from swarm_collective.utils import plot_communication_graph
from IPython.display import Image, display

obs_list = [obs_dict[i] for i in range(len(env.role_ids))]
obs_g = torch.stack([torch.from_numpy(o["global"]).float() for o in obs_list]).to(device)
obs_l = torch.stack([torch.from_numpy(o["local"]).float() for o in obs_list]).to(device)
ctx   = torch.stack([torch.from_numpy(o["ctx"]).float() for o in obs_list]).to(device)
roles = torch.tensor(env.role_ids, dtype=torch.long, device=device)
pos   = torch.tensor([list(p) for p in env.agent_pos], dtype=torch.long, device=device)
alive = torch.tensor(list(env.agent_alive), dtype=torch.bool, device=device)

with torch.no_grad():
    feats = system.encode_all(obs_g, obs_l, ctx, roles)
    q_vals, comm_penalty, attn = system.communicate_and_qvals(feats, pos, alive, comm_range=env.comm_range)

print(f"Q-values shape      : {q_vals.shape} (N_agents, 4 actions)")
print(f"Comm penalty applied: {comm_penalty.item():.4f}")
print(f"Attention weights (N×N):\n{attn.cpu().numpy()}")

# Render spatial graph
plot_communication_graph(
    positions=[list(p) for p in env.agent_pos],
    role_names=[ROLE_NAMES[r] for r in env.role_ids],
    alive_mask=list(env.agent_alive),
    attn_matrix=attn.cpu().numpy(),
    target_pos=list(env.target_pos),
    save_path="demo_comm_graph.png"
)

display(Image("demo_comm_graph.png"))

## 🎓 Cell 5: Full 5-Stage Curriculum Training
Runs the complete QMIX + TarMAC training loop.

| Stage | Name | Team Composition | Mission | Advance Criteria |
|---|---|---|---|---|
| 1 | `scout_only` | 3 Scouts | Target Search | Success Rate $\ge 70\%$ |
| 2 | `scout_carrier` | 2 Scouts + 1 Carrier | Search & Transport | Success Rate $\ge 65\%$ |
| 3 | `scout_analyzer` | 2 Scouts + 1 Analyzer | Search & Analysis | Success Rate $\ge 60\%$ |
| 4 | `full_swarm` | 2 Scouts + 1 Carrier + 1 Analyzer | Full Pipeline | Success Rate $\ge 55\%$ |
| 5 | `full_swarm_faults` | Full Swarm | Mission with Random Failures | Continuous |

*(Adjust `total_episodes` as desired; e.g. 500 for a quick demo or 5000+ for deeper convergence)*

In [ ]:
from swarm_collective.train import run_training, load_config

config = load_config("swarm_collective/configs/swarm_config.yaml")

# Configure training budget
TOTAL_EPISODES = 500   # Set to 1000 - 5000 for full convergence

history = run_training(
    config=config,
    total_episodes=TOTAL_EPISODES,
    pretrained_path=ckpt_path if os.path.exists(ckpt_path) else None,
    device_str="cuda" if torch.cuda.is_available() else "cpu"
)

## 📊 Cell 6: Visualizing Training Curves & Metrics
Displays the multi-panel training telemetry:
1. **Rolling Success Rate**
2. **Team Cumulative Reward**
3. **Communication Volume** $\|m\|_2$ (shows how the bandwidth penalty suppresses wasteful chatter)
4. **QMIX Joint TD Loss**

In [ ]:
from IPython.display import Image, display

if os.path.exists("training_curves.png"):
    print("=== Training Progress Telemetry ===")
    display(Image("training_curves.png"))

if os.path.exists("last_comm_graph.png"):
    print("=== Final Evaluation Communication Graph ===")
    display(Image("last_comm_graph.png"))

## 🔬 Cell 7: Ablation Study — Communicating Swarm vs. Silent Swarm
Runs an evaluation benchmark comparing:
- **Collective Intelligence (TarMAC Enabled)**: Drones broadcast messages and route target discoveries.
- **Isolated Swarm (Comms Disabled)**: Drones operate blindly without exchanging signals.

Quantifies the performance gain of learned communication.

In [ ]:
def evaluate_swarm(env, system, n_eval_episodes=50, allow_comm=True):
    successes = 0
    total_steps = []
    
    for _ in range(n_eval_episodes):
        obs_dict, info = env.reset()
        obs_list = [obs_dict[i] for i in range(len(env.role_ids))]
        steps = 0
        terminated = truncated = False
        
        while not (terminated or truncated):
            obs_g = torch.stack([torch.from_numpy(o["global"]).float() for o in obs_list]).to(device)
            obs_l = torch.stack([torch.from_numpy(o["local"]).float() for o in obs_list]).to(device)
            ctx   = torch.stack([torch.from_numpy(o["ctx"]).float() for o in obs_list]).to(device)
            roles = torch.tensor(env.role_ids, dtype=torch.long, device=device)
            pos   = torch.tensor([list(p) for p in env.agent_pos], dtype=torch.long, device=device)
            alive = torch.tensor(list(env.agent_alive), dtype=torch.bool, device=device)
            
            with torch.no_grad():
                feats = system.encode_all(obs_g, obs_l, ctx, roles)
                if allow_comm:
                    q_vals, _, _ = system.communicate_and_qvals(feats, pos, alive, comm_range=env.comm_range)
                else:
                    # Zero out received context (no communication)
                    zero_ctx = torch.zeros(feats.size(0), system.agent.tarmac.dim_msg, device=device)
                    q_vals = system.agent.compute_q(feats, zero_ctx)
            
            actions = [int(q_vals[i].argmax().item()) if env.agent_alive[i] else 0 for i in range(len(env.role_ids))]
            next_obs_dict, rewards, terminated, truncated, next_info = env.step(actions)
            obs_list = [next_obs_dict[i] for i in range(len(env.role_ids))]
            steps += 1
        
        if next_info.get("task_phase") == "done" or next_info.get("target_found", False):
            successes += 1
        total_steps.append(steps)
        
    return successes / n_eval_episodes, np.mean(total_steps)

print("Evaluating Collective Swarm with TarMAC...")
comm_sr, comm_steps = evaluate_swarm(env, system, n_eval_episodes=30, allow_comm=True)

print("Evaluating Ablated Swarm without Communication...")
silent_sr, silent_steps = evaluate_swarm(env, system, n_eval_episodes=30, allow_comm=False)

print("\n=================== 📊 ABLATION RESULTS ===================")
print(f"  TarMAC Collective Swarm : Success Rate = {comm_sr*100:5.1f}% | Avg Steps = {comm_steps:.1f}")
print(f"  Isolated (Silent) Swarm : Success Rate = {silent_sr*100:5.1f}% | Avg Steps = {silent_steps:.1f}")
print(f"  Gain from Communication : +{(comm_sr - silent_sr)*100:5.1f}% Success Rate")
print("===========================================================")